**02 · Campo de Christoffel y estimación de soporte**

Ejecutar desde el repositorio point-field con `python -m pip install -e ".[learning,bench,dev]"` y Jupyter. Las salidas nuevas se guardan en `runs/tesis_figuras/`. No reemplazar resultados históricos sin comprobar el protocolo.

In [ ]:
from pathlib import Path
import sys, subprocess, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None):
    if y is None: ax.scatter(*X.T, s=5, c=COLORS[0], linewidths=0)
    else:
        for j,c in enumerate(np.unique(y)):
            ax.scatter(*X[y==c].T, s=5, c='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)], linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2: ax.set_aspect('equal', adjustable='box')
    else: ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, '\nSalidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except (OSError, subprocess.CalledProcessError):
    commit='No disponible'
print('Commit:',commit)

**1. Campo escalar.** Ajustar CD a las nubes de point-field; el umbral del percentil 95 de entrenamiento es ilustrativo y no constituye una estimación independiente de cobertura.

In [ ]:
from pointfield.geometry.shapes2d import Disk,Annulus,Ellipse,Rectangle
from pointfield.learning import ChristoffelDarboux
shapes=[('Disco',Disk()),('Anillo',Annulus(.5,1)),('Elipse',Ellipse(1,.5)),('Rectángulo',Rectangle(2,1))]
fig,axs=plt.subplots(len(shapes),4,figsize=(10,9),constrained_layout=True)
for i,(name,g) in enumerate(shapes):
    X=g.sample(500,seed=100); lo=X.min(0)-.3; hi=X.max(0)+.3
    a,b=np.meshgrid(np.linspace(lo[0],hi[0],150),np.linspace(lo[1],hi[1],150)); G=np.c_[a.ravel(),b.ravel()]
    m=ChristoffelDarboux(degree=4,reg=1e-8).fit(X)
    q=np.asarray(m.score(G)).reshape(a.shape); threshold=np.quantile(m.score(X),.95)
    for ax in axs[i]: ax.set(xlim=(lo[0],hi[0]),ylim=(lo[1],hi[1]),xticks=[],yticks=[]);ax.set_aspect('equal')
    points(axs[i,0],X);axs[i,0].set_ylabel(name)
    axs[i,1].contourf(a,b,np.log10(np.maximum(q,1e-15)),levels=24,cmap='viridis');axs[i,1].scatter(*X.T,s=1,c='white')
    points(axs[i,2],X);axs[i,2].contour(a,b,q,levels=[threshold],colors='#c65036')
    axs[i,3].contourf(a,b,(q<=threshold).astype(float),levels=[.5,1.5],colors=['#a9c9df'])
for ax,title in zip(axs[0],['Nube','log10 q','Contorno q=umbral','Soporte estimado']):ax.set_title(title)
save(fig,'02_campo_soporte')

**2. Grado y tamaño muestral.** Utilizar los CSV del diagnóstico independiente incluido en el paquete de Overleaf o regenerarlos con el script autónomo adjunto. Las figuras resultantes no se confunden con E1 de point-field.

In [ ]:
diag=next((d for d in [ROOT/'notebooks'/'diagnostico_grado_muestras.py',ROOT/'notebooks_tesis'/'diagnostico_grado_muestras.py',Path.cwd()/'diagnostico_grado_muestras.py'] if d.exists()),ROOT/'notebooks'/'diagnostico_grado_muestras.py')
if not diag.exists():
    diag=Path.cwd()/'diagnostico_grado_muestras.py'
if diag.exists():
    import importlib.util
    spec=importlib.util.spec_from_file_location('diagnostico_tesis',diag); module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    module.OUT=OUT;module.RES=OUT;module.main()
    display(pd.read_csv(OUT/'diagnostico_resumen.csv'))
else:
    print('Copia diagnostico_grado_muestras.py del paquete a point-field/notebooks y ejecuta esta celda de nuevo.')